# Day 52 — Solutions: Scalability with Dask

We port a pandas pipeline to Dask, experiment with chunk sizes, and use the dashboard to find bottlenecks.

Contents
- Exercise 1: Convert pandas pipeline to Dask and benchmark
- Exercise 2: Adjust chunk sizes and measure impact
- Exercise 3: Use dashboard to identify bottlenecks

---

Exercise 1 — Port pandas to Dask

> Generated from `python/ds-60day/solutions/day52_scalability_dask/day52_solutions.md` by `scripts/build_solution_notebooks.py`. Edit the Markdown source, then regenerate this notebook.


In [ ]:
import dask.dataframe as dd
import dask
from dask.diagnostics import ProgressBar

# Example: read many Parquet files
path = 's3://bucket/data/*.parquet'  # or local pattern

ddf = dd.read_parquet(path)

# Pipeline: filter → feature → groupby aggregate
pipe = (ddf[ddf.amount > 0]
          .assign(unit=lambda d: d.amount/d.qty)
          .groupby('store').unit.mean())

with ProgressBar():
    res = pipe.compute()
print(res.head())


Explanation
- dd.read_parquet lazily constructs a task graph
- compute triggers execution on the scheduler
- ProgressBar provides visibility for CLI runs

---

Exercise 2 — Chunk sizes


In [ ]:
import dask.array as da
import time

# About 128 MB of logical float64 data: visible chunking without a 12.8 GB job.
shape = (4_000, 4_000)
for chunk in [(500, 500), (1_000, 1_000), (2_000, 2_000)]:
    x = da.random.random(shape, chunks=chunk)
    t0 = time.perf_counter()
    x.mean().compute()
    elapsed = time.perf_counter() - t0
    print({'chunk': chunk, 'seconds': round(elapsed, 2)})


Notes
- Too small chunks → overhead; too large → poor parallelism/memory pressure
- Use dashboard + this laptop-safe microbenchmark before increasing the shape

---

Exercise 3 — Dashboard


In [ ]:
from dask.distributed import Client

client = Client()  # local cluster; dashboard at http://localhost:8787
print(client)

# Persist intermediate to keep in memory across steps
cached = ddf.persist()


Guidance
- Use the dashboard’s Task Stream to see long‑running tasks
- Profile memory to detect spilling; consider repartitioning: `ddf = ddf.repartition(npartitions=desired)`
- Close client when done to free resources: `client.close()`

---

## Exercise-by-exercise reasoning map

This map connects every learner prompt to a reasoning path. Read the
explanation before copying code: the goal is to understand the assumptions,
the evidence that validates the result, and the edge cases that can make an
apparently correct implementation fail.

### Exercise 1 — Original lesson practice

**Prompt:** Read a large local CSV with Dask and compute groupby aggregations.

**How to reason about it:** Generate or use a local CSV, begin small, and compare Dask aggregation with pandas before scaling. Dask expressions are lazy until `compute`, so measure materialization separately.

Use the worked reference earlier in this file, then change one boundary
condition and rerun the stated checks. A copied output is not evidence
unless you can explain why that output follows from the inputs.

### Exercise 2 — Original lesson practice

**Prompt:** Persist the DataFrame and compare repeated timings with and without persistence.

**How to reason about it:** Persist is beneficial only for reused intermediates and consumes memory. Time a first materialization plus repeated downstream actions, then close distributed clients and release references.

Use the worked reference earlier in this file, then change one boundary
condition and rerun the stated checks. A copied output is not evidence
unless you can explain why that output follows from the inputs.

### Exercise 3 — Original lesson practice

**Prompt:** Implement the same reduction with `pandas.read_csv(..., chunksize=...)` and compare memory and elapsed time.

**How to reason about it:** Chunked pandas should carry partial sums and counts rather than concatenate all chunks. Verify the final reduction against an in-memory result and handle missing/group-key semantics consistently.

Use the worked reference earlier in this file, then change one boundary
condition and rerun the stated checks. A copied output is not evidence
unless you can explain why that output follows from the inputs.

### Exercise 4 — Task-graph tracing

**Prompt:** Build two aggregations from the same lazy Dask DataFrame, inspect their task graphs, and compare separate computes with one combined `dask.compute` call.

**Reasoning before implementation:** Building an expression does not read all data. Combining terminal computations can share upstream work without persisting the entire frame.

Record graph/task counts and wall time, but treat timings as local evidence
rather than universal benchmarks. `dask.compute(result_a, result_b)` can
optimize shared dependencies in one execution. Two separate `.compute()` calls
may reread and redo work unless data or intermediates are cached/persisted.

Keep output bounded; computing a small aggregation is different from calling
`compute()` on the full large DataFrame.

**Why this matters:** The result should survive a fresh-kernel rerun and
a deliberately chosen boundary case. If it does not, revisit the
assumption or data boundary rather than hiding the failure.

### Exercise 5 — Partition-skew diagnosis

**Prompt:** Create a group key where one value owns most rows. Measure partition sizes and groupby runtime, then propose repartitioning or algorithm changes.

**Reasoning before implementation:** A balanced row count before a shuffle does not guarantee balanced work after grouping; one hot key can become a straggler.

Report the largest group and partition relative to the median. Salting a hot
key can distribute an associative partial aggregation, followed by a second
combine, but it changes implementation complexity. More partitions alone do
not split one indivisible group operation automatically.

If the output is tiny, pre-aggregate within input partitions before the
shuffle. Preserve exactness and test the salted/two-stage result against pandas.

**Why this matters:** The result should survive a fresh-kernel rerun and
a deliberately chosen boundary case. If it does not, revisit the
assumption or data boundary rather than hiding the failure.

### Exercise 6 — Reducer correctness

**Prompt:** Implement a mergeable mean/variance state for chunks and prove it matches NumPy across different chunk boundaries, including an empty chunk.

**Reasoning before implementation:** A mean alone is not mergeable without support. Carry count, mean, and M2 (sum of squared deviations) using a stable combine formula.

The state must define an identity for an empty chunk and an associative combine
operation (within floating-point tolerance). Welford/Chan formulas combine
`count`, `mean`, and `M2` without concatenating values.

Test one chunk, one-row chunks, uneven chunks, large-offset values, and an
empty input policy. This same reasoning underlies correct distributed custom
aggregations; averaging partition means is wrong when partition sizes differ.

**Why this matters:** The result should survive a fresh-kernel rerun and
a deliberately chosen boundary case. If it does not, revisit the
assumption or data boundary rather than hiding the failure.
